<a href="https://colab.research.google.com/github/YoussefAli07/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
!git clone https://github.com/YoussefAli07/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [30]:
import pandas as pd
df_raw = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df_raw.shape)
df_raw.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [32]:
import os

# Load the raw dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Rebuild ctr_gap (ML-02/ML-07 logic)
df['position_avg_ctr'] = df.groupby('position_tier')['ctr'].transform('mean')
df['ctr_gap'] = df['ctr'] - df['position_avg_ctr']

# The three ML-07 gates
visible = df['impressions_90d'] >= 81
stale = df['freshness_tier'] == '181+'
underperforming = df['ctr_gap'] <= -0.5

flagged = df[visible & stale & underperforming].copy()

# Score + rank
flagged['score'] = flagged['ctr_gap'].abs() * flagged['impressions_90d']
flagged = flagged.sort_values('score', ascending=False).reset_index(drop=True)

# Reason code + action label (from your ML-07 write-up)
flagged['reason_code'] = 'zero_clicks_visible_but_stale'
flagged['action_label'] = 'Needs review — potentially a title/meta rewrite'

print(flagged.shape)
flagged[['content_id', 'score', 'impressions_90d', 'action_label']]

(9, 49)


,content_id,score,impressions_90d,action_label
0,content_fd16e3475c29,279.908156,429,Needs review — potentially a title/meta rewrite
1,content_ea41fe5cf292,172.903640,265,Needs review — potentially a title/meta rewrite
2,content_958a46db26bd,129.188380,198,Needs review — potentially a title/meta rewrite
3,content_02b0d6e30129,114.834115,176,Needs review — potentially a title/meta rewrite
4,content_f488400fca67,101.132318,155,Needs review — potentially a title/meta rewrite
5,content_ab27c30d81f4,67.204056,103,Needs review — potentially a title/meta rewrite
6,content_07ce98c6085a,55.459658,85,Needs review — potentially a title/meta rewrite
7,content_30eb41dff556,54.807191,84,Needs review — potentially a title/meta rewrite
8,content_460b11dcac6a,52.849792,81,Needs review — potentially a title/meta rewrite


In [33]:
def build_reason_code(row):
    tags = []
    if row['impressions_90d'] >= 81:
        tags.append('Visible')
    if row['freshness_tier'] == '181+':
        tags.append('Stale')
    if row['ctr_gap'] <= -0.5:
        tags.append('Low CTR')
    return ', '.join(tags) if tags else 'No gate triggered'

flagged['reason_code_v2'] = flagged.apply(build_reason_code, axis=1)
flagged[['content_id', 'reason_code_v2', 'action_label']]

,content_id,reason_code_v2,action_label
0,content_fd16e3475c29,"Visible, Stale, Low CTR",Needs review — potentially a title/meta rewrite
1,content_ea41fe5cf292,"Visible, Stale, Low CTR",Needs review — potentially a title/meta rewrite
2,content_958a46db26bd,"Visible, Stale, Low CTR",Needs review — potentially a title/meta rewrite
3,content_02b0d6e30129,"Visible, Stale, Low CTR",Needs review — potentially a title/meta rewrite
4,content_f488400fca67,"Visible, Stale, Low CTR",Needs review — potentially a title/meta rewrite
5,content_ab27c30d81f4,"Visible, Stale, Low CTR",Needs review — potentially a title/meta rewrite
6,content_07ce98c6085a,"Visible, Stale, Low CTR",Needs review — potentially a title/meta rewrite
7,content_30eb41dff556,"Visible, Stale, Low CTR",Needs review — potentially a title/meta rewrite
8,content_460b11dcac6a,"Visible, Stale, Low CTR",Needs review — potentially a title/meta rewrite


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.